In [ ]:

#####################################################
#                                                   #
#  Helper functions and includes. RUN THIS FIRST!!  #
#                                                   #
#####################################################


from pynq import Overlay, allocate, MMIO
from pynq.lib.video import DisplayPort, VideoMode, PIXEL_RGB
import numpy as np
import matplotlib.pyplot as plt
import time
from pynq.lib import AxiIIC
import cv2
import numpy as np


def write_cam_reg(reg, val):
    reg_msb = (reg >> 8) & 0xFF
    reg_lsb = reg & 0xFF
    
    # Send as a 3-byte payload: [Address High, Address Low, Data]
    iic.send(CAM_I2C_ADDR, [reg_msb, reg_lsb, val], length=3)

def read_cam_reg(reg):
    reg_msb = (reg >> 8) & 0xFF
    reg_lsb = reg & 0xFF
    
    iic.send(CAM_I2C_ADDR, [reg_msb, reg_lsb], length=2)

    val = iic.receive(CAM_I2C_ADDR, length=1)
    return val[0]


def _RUN_PIPELINE_DIAGNOSTICS():
    print("\n--- PIPELINE DIAGNOSTICS ---")

    # 1. MIPI IP Diagnostics
    mipi = MMIO(ol.ip_dict['mipi_csi2_rx_subsyst_0']['phys_addr'], 0x1000)
    mipi_status = mipi.read(0x10)
    mipi_intr = mipi.read(0x24)
    mipi_pkts = mipi.read(0x60)
    mipi_err1 = mipi.read(0x08)
    mipi_err2 = mipi.read(0x0c)

    print("MIPI CSI-2 Rx:")
    print(f"  Core Status (0x10):   {hex(mipi_status)}")
    print(f"  Interrupts (0x24):    {hex(mipi_intr)}")
    print(f"  Packet Count (0x60):  {mipi_pkts} (If 0, camera is asleep/broken)")

    # 2. Demosaic IP Diagnostics
    demo_ctrl = demosaic.read(0x00)
    print("\nDemosaic IP:")
    print(f"  Control Reg (0x00):   {hex(demo_ctrl)}")
    if demo_ctrl == 0x81:
        print("  -> ERROR: Stuck waiting for Start of Frame! (Resolution mismatch?)")
    elif demo_ctrl & 0x02:
        print("  -> SUCCESS: AP_DONE flag is set, Demosaic processed a frame.")

    # 3. VDMA Diagnostics
    vdma_status = vdma.read(0x34)
    print("\nVDMA Write (S2MM):")
    print(f"  Status Reg (0x34):    {hex(vdma_status)}")
    if vdma_status & 0x1000:
        print("  -> NOTE: FrmCnt_Irq (Running fine tho probably)")
    elif vdma_status & 0x4000:
        print("  -> ERROR: Start of Frame Early (Stream out of sync)")
    elif vdma_status == 0x10000:
        print("  -> ERROR: No data received/idling.")
    print("----------------------------\n")
    
    
    
    
# DISCLAIMER DISCLAIMER DISCLAIMER I DIDN'T WRITE ALL OF THESE

# Adapted/converted from https://github.com/Digilent/Zybo-Z7-SW/blob/5d358aa3469869382f9fb515b65af3469775e3c8/src/Zybo-Z7-20-pcam-5c/src/ov5640/OV5640.h
# Full credit to digilent and their work on the Zybo Z7-20 and OV5640 camera module. All I've done here is convert the register mappings to 
# be python compatible.

    
cfg_1080p_30fps_336M_mipi = [
    #1920 x 1080 @ 30fps, RAW10, MIPISCLK=672, SCLK=67.2MHz, PCLK=134.4M
    #PLL1 configuration
    #[7:4]=0001 System clock divider /1, [3:0]=0001 Scale divider for MIPI /1
    (0x3035, 0x11), # 30fps setting
    #[7:0]=84 PLL multiplier
    (0x3036, 0x54),
    #[4]=1 PLL root divider /2, [3:0]=5 PLL pre-divider /1.5
    (0x3037, 0x15),
    #[5:4]=00 PCLK root divider /1, [3:2]=00 SCLK2x root divider /1, [1:0]=01 SCLK root divider /2
    (0x3108, 0x01),

    #[6:4]=001 PLL charge pump, [3:0]=1010 MIPI 10-bit mode
    (0x3034, 0x1A),

    #[3:0]=0 X address start high byte
    (0x3800, (336 >> 8) & 0x0F),
    #[7:0]=0 X address start low byte
    (0x3801, 336 & 0xFF),
    #[2:0]=0 Y address start high byte
    (0x3802, (426 >> 8) & 0x07),
    #[7:0]=0 Y address start low byte
    (0x3803, 426 & 0xFF),

    #[3:0] X address end high byte
    (0x3804, (2287 >> 8) & 0x0F),
    #[7:0] X address end low byte
    (0x3805, 2287 & 0xFF),
    #[2:0] Y address end high byte
    (0x3806, (1529 >> 8) & 0x07),
    #[7:0] Y address end low byte
    (0x3807, 1529 & 0xFF),

    #[3:0]=0 timing hoffset high byte
    (0x3810, (16 >> 8) & 0x0F),
    #[7:0]=0 timing hoffset low byte
    (0x3811, 16 & 0xFF),
    #[2:0]=0 timing voffset high byte
    (0x3812, (12 >> 8) & 0x07),
    #[7:0]=0 timing voffset low byte
    (0x3813, 12 & 0xFF),

    #[3:0] Output horizontal width high byte
    (0x3808, (1920 >> 8) & 0x0F),
    #[7:0] Output horizontal width low byte
    (0x3809, 1920 & 0xFF),
    #[2:0] Output vertical height high byte
    (0x380a, (1080 >> 8) & 0x7F),
    #[7:0] Output vertical height low byte
    (0x380b, 1080 & 0xFF),

    #HTS line exposure time in # of pixels Tline=HTS/sclk
    (0x380c, (2500 >> 8) & 0x1F),
    (0x380d, 2500 & 0xFF),
    #VTS frame exposure time in # lines
    (0x380e, (1120 >> 8) & 0xFF),
    (0x380f, 1120 & 0xFF),
    
    #Change exposure auto settings
    (0x3503, 0x06), # 0x06 == Turn off AEC and AGC
    
    # Manual Exposure
    (0x3500, 0x00),
    (0x3501, 0x38), # Was 0x30. 0xA0 will keep the shutter open much longer.
    (0x3502, 0x00),
    
    # Manual Analog Gain                          !!!!!!!!!!!
    (0x350A, 0x08), # High Byte of Gain
    (0x350B, 0x80), # Low Byte of Gain\
    
    

    #[7:4]=0x1 horizontal odd subsample increment, [3:0]=0x1 horizontal even subsample increment
    (0x3814, 0x11),
    #[7:4]=0x1 vertical odd subsample increment, [3:0]=0x1 vertical even subsample increment
    (0x3815, 0x11),

    #[2]=0 ISP mirror, [1]=0 sensor mirror, [0]=0 no horizontal binning
    (0x3821, 0x00),

    #little MIPI shit: global timing unit, period of PCLK in ns * 2(depends on # of lanes)
    (0x4837, 14), # 1/84M*2

    #Undocumented anti-green settings
    (0x3618, 0x00), # Removes vertical lines appearing under bright light
    (0x3612, 0x59),
    (0x3708, 0x64),
    (0x3709, 0x52),
    (0x370c, 0x03),

    #[7:4]=0x0 Formatter RAW, [3:0]=0x0 BGBG/GRGR
    (0x4300, 0x00),
    #[2:0]=0x3 Format select ISP RAW (DPC)
    (0x501f, 0x03)
]





cfg_init = [
    #[7]=0 Software reset; [6]=1 Software power down; Default=0x02
    (0x3008, 0x42),
    #[1]=1 System input clock from PLL; Default read = 0x11
    (0x3103, 0x03),
    #[3:0]=0000 MD2P,MD2N,MCP,MCN input; Default=0x00
    (0x3017, 0x00),
    #[7:2]=000000 MD1P,MD1N, D3:0 input; Default=0x00
    (0x3018, 0x00),
    #[6:4]=001 PLL charge pump, [3:0]=1000 MIPI 8-bit mode
    (0x3034, 0x18),

    #PLL1 configuration
    #[7:4]=0001 System clock divider /1, [3:0]=0001 Scale divider for MIPI /1
    (0x3035, 0x11),
    #[7:0]=56 PLL multiplier
    (0x3036, 0x38),
    #[4]=1 PLL root divider /2, [3:0]=1 PLL pre-divider /1
    (0x3037, 0x11),
    #[5:4]=00 PCLK root divider /1, [3:2]=00 SCLK2x root divider /1, [1:0]=01 SCLK root divider /2
    (0x3108, 0x01),
    #PLL2 configuration
    #[5:4]=01 PRE_DIV_SP /1.5, [2]=1 R_DIV_SP /1, [1:0]=00 DIV12_SP /1
    (0x303D, 0x10),
    #[4:0]=11001 PLL2 multiplier DIV_CNT5B = 25
    (0x303B, 0x19),

    (0x3630, 0x2e),
    (0x3631, 0x0e),
    (0x3632, 0xe2),
    (0x3633, 0x23),
    (0x3621, 0xe0),
    (0x3704, 0xa0),
    (0x3703, 0x5a),
    (0x3715, 0x78),
    (0x3717, 0x01),
    (0x370b, 0x60),
    (0x3705, 0x1a),
    (0x3905, 0x02),
    (0x3906, 0x10),
    (0x3901, 0x0a),
    (0x3731, 0x02),
    #VCM debug mode
    (0x3600, 0x37),
    (0x3601, 0x33),
    #System control register changing not recommended
    (0x302d, 0x60),
    #??
    (0x3620, 0x52),
    (0x371b, 0x20),
    #?? DVP
    (0x471c, 0x50),

    (0x3a13, 0x43),
    (0x3a18, 0x00),
    (0x3a19, 0xf8),
    (0x3635, 0x13),
    (0x3636, 0x06),
    (0x3634, 0x44),
    (0x3622, 0x01),
    (0x3c01, 0x34),
    (0x3c04, 0x28),
    (0x3c05, 0x98),
    (0x3c06, 0x00),
    (0x3c07, 0x08),
    (0x3c08, 0x00),
    (0x3c09, 0x1c),
    (0x3c0a, 0x9c),
    (0x3c0b, 0x40),

    #[7]=1 color bar enable, [3:2]=00 eight color bar
    (0x503d, 0x00),
    #[2]=1 ISP vflip, [1]=1 sensor vflip
    (0x3820, 0x46),

    #[7:5]=010 Two lane mode, [4]=0 MIPI HS TX no power down, [3]=0 MIPI LP RX no power down, [2]=1 MIPI enable, [1:0]=10 Debug mode; Default=0x58
    (0x300e, 0x45),
    #[5]=0 Clock free running, [4]=1 Send line short packet, [3]=0 Use lane1 as default, [2]=1 MIPI bus LP11 when no packet; Default=0x04
    (0x4800, 0x14),
    (0x302e, 0x08),
    #[7:4]=0x3 YUV422, [3:0]=0x0 YUYV
    #(0x4300, 0x30),
    #[7:4]=0x6 RGB565, [3:0]=0x0 (b[4:0],g[5:3],g[2:0],r[4:0])
    (0x4300, 0x6f),
    (0x501f, 0x01),

    (0x4713, 0x03),
    (0x4407, 0x04),
    (0x440e, 0x00),
    (0x460b, 0x35),
    #[1]=0 DVP PCLK divider manual control by 0x3824[4:0]
    (0x460c, 0x20),
    #[4:0]=1 SCALE_DIV=INT(3824[4:0]/2)
    (0x3824, 0x01),

    #MIPI timing
    #		(0x4805, 0x10), #LPX global timing select=auto
    #		(0x4818, 0x00), #hs_prepare + hs_zero_min ns
    #		(0x4819, 0x96),
    #		(0x482A, 0x00), #hs_prepare + hs_zero_min UI
    #
    #		(0x4824, 0x00), #lpx_p_min ns
    #		(0x4825, 0x32),
    #		(0x4830, 0x00), #lpx_p_min UI
    #
    #		(0x4826, 0x00), #hs_prepare_min ns
    #		(0x4827, 0x32),
    #		(0x4831, 0x00), #hs_prepare_min UI

    #[7]=1 LENC correction enabled, [5]=1 RAW gamma enabled, [2]=1 Black pixel cancellation enabled, [1]=1 White pixel cancellation enabled, [0]=1 Color interpolation enabled
    (0x5000, 0x07),
    #[7]=0 Special digital effects, [5]=0 scaling, [2]=0 UV average disabled, [1]=1 Color matrix enabled, [0]=1 Auto white balance enabled
    (0x5001, 0x03)
]



# Overlay wrapper with a little extra functionality 
def _LOAD_OVERLAY(ol_file):
    print(f"Loading bitstream: {ol_file}")
    ol = Overlay(ol_file)
    
    print("Overlay loaded. Printing IPs detected for debug.")

    try: 
        for ip_name, ip_info in ol.ip_dict.items():
            ip_type = ip_info.get('type', 'Unknown Type')
            phys_addr = hex(ip_info.get('phys_addr', 0))
            print(f"{ip_name:<30} | {ip_type:<40} | {phys_addr}")

    except Exception as e:
        print(f"An unexpected error occurred: {e}")
    
    print ("\nOverlay initialized and IP printed (probably)\n\n")
    
    return ol



# ENSURE I2C IS ENABLED FOR ALL PCAM OPERATIONS
# built in delays for safety for all PCAM operations
# must be run first to ensure camera is operating properly
def _PCAM_CONFIGURE_BASE():
    for reg, val in cfg_init:
        write_cam_reg(reg, val)
        time.sleep(0.005)
    time.sleep(0.5)
    print("PCAM5 Initialized.")
    
# configure standard resolution
def _PCAM_CONFIGURE_1080p30():
    for reg, val in cfg_1080p_30fps_336M_mipi:
        write_cam_reg(reg, val)
        time.sleep(0.005)
    time.sleep(0.5)
    print("PCAM5 resolutions and format set to 1920*1080@30fps over RAW10.\n")

# Powercycle pcam
def _PCAM_PC(pwup_pin):
    print("Powering down camera...")
    pwup_pin.write(0) 
    time.sleep(0.1)

    print("Powering up camera...")
    pwup_pin.write(1)
    time.sleep(.5)
    
def _PCAM_START_CAPTURE():
    write_cam_reg(0x3008, 0x02) #starts the cam - MUST BE THE LAST COMMAND ISSUED... heavens this caused so many issues...
    time.sleep(0.5)
    print("PCAM5 started. Delaying to allow settling, flush=True")    
    
    
    ###########
# DEMOSAIC CONTROL #
    ###########
def _DEMOSAIC_CONFIG_RES(ol_id, width, height):
    ol_id.write(0x10, width)     # 0x10  Width reg
    ol_id.write(0x18, height)     # 0x18  Height reg
    print(f"Demosaic set to {width}x{height} resolution\n")
    
def _DEMOSAIC_SET_BAYER(ol_id, bayer_pattern=0x00): #0x00 should be correct for pcam5 performed a "GSS" to figure this out :p
    ol_id.write(0x28, bayer_pattern)
    print(f"Demosaic set to Bayer pattern {bayer_pattern}\n")
    
def _DEMOSAIC_SET_CTRL(ol_id, val=0x81):
    ol_id.write(0x00, val) 
    print(f"Demosaic Control Reg Status: {hex(ol_id.read(0x00))}")
    
    
    
    
    
def _VDMA_CONFIG_WHOLISTIC(ol_id, fb1, fb2, fb3, fb4, width, height):
    print("Configuring VDMA", flush=True)
    stride = width * 3  # 3 bytes per pixel for RGB888
    ol_id.write(0x30, 0x4)        # Soft Reset

    timeout = 20
    while (ol_id.read(0x30) & 0x04) == 0x04:
        time.sleep(0.05) 
        timeout -= 1
        if timeout == 0:
            print("CRITICAL: VDMA Reset stuck!")
            break

    print("Reset settled. Arming VDMA.")
    ol_id.write(0x30, 0x3)        # Start S2MM, Circular Mode

    # VDAM is configured for 4 frame buffers in my version. adjust as needed
    ol_id.write(0xAC, fb1.device_address) 
    ol_id.write(0xB0, fb2.device_address) 
    ol_id.write(0xB4, fb3.device_address)  
    ol_id.write(0xB8, fb4.device_address)
    
    
    # Set sizes
    ol_id.write(0xA8, stride)                      
    ol_id.write(0xA4, stride)                      
    ol_id.write(0xA0, height)
    print("VDMA Sizes set...")

    time.sleep(0.5)
    status = ol_id.read(0x34)
    print(f" -> VDMA Armed. Status: {hex(status)}", flush=True)
    if status == 0x10000:
        print(" -> SUCCESS: VDMA is perfectly waiting for the camera!", flush=True)

def _ALLOCATE_FRAME_BUFFER(width, height):
    print(f"Allocating {width}x{height} frame buffer")
    fb = allocate(shape=(height, width, 3), dtype=np.uint8)

    print("Frame buffer allocated of requested size. Clearing buffer to gray for debugging.")
    fb[:] = 128
    fb.flush()
    print(f" -> Buffer physical address: {hex(fb.device_address)}\n", flush=True)
    return fb


def cap_latest_frame(vdma):
    ########################### Get the oldest frame ########################## put this in a frame to collect frames continuously
    oldest_frame = (((vdma.read(0x28))>>24) - 1) % 4


    #flush

    rgb_frame = None

    if oldest_frame == 0:
        frame_buffer1.invalidate()
        return cv2.cvtColor(frame_buffer1, cv2.COLOR_BGR2RGB)
        
    elif oldest_frame == 1:
        frame_buffer2.invalidate()
        return cv2.cvtColor(frame_buffer2, cv2.COLOR_BGR2RGB)
        
    elif oldest_frame == 2:
        frame_buffer3.invalidate()
        return cv2.cvtColor(frame_buffer3, cv2.COLOR_BGR2RGB)
        
    else:
        frame_buffer4.invalidate()
        return cv2.cvtColor(frame_buffer4, cv2.COLOR_BGR2RGB)
    ############################

    
def display_bounding_box_1080():
    dp_frame = dp.newframe()
    
    oldest_frame = (((vdma.read(0x28))>>24) - 1) % 2
#     print("Drawing frame: " + str(oldest_frame))
    
    if(oldest_frame == 0):
        frame_buffer_color1.invalidate()
        np.copyto(dp_frame, frame_buffer_color1)
    else:
        frame_buffer_color2.invalidate()
        np.copyto(dp_frame, frame_buffer_color2)

    dp.writeframe(dp_frame)
    
    
    
def cap_latest_frame(vdma):
    ########################### Get the oldest frame ########################## put this in a frame to collect frames continuously
    oldest_frame = (((vdma.read(0x28))>>24) - 1) % 4


    #flush

    rgb_frame = None

    if(oldest_frame == 0):
        frame_buffer1.invalidate()
        rgb_frame = np.copy(frame_buffer1)[:, :, ::-1]
    elif (oldest_frame == 1):
        frame_buffer2.invalidate()
        rgb_frame = np.copy(frame_buffer2)[:, :, ::-1]
    elif (oldest_frame == 1):
        frame_buffer3.invalidate()
        rgb_frame = np.copy(frame_buffer3)[:, :, ::-1]
    else:
        frame_buffer4.invalidate()
        rgb_frame = np.copy(frame_buffer4)[:, :, ::-1]

    return rgb_frame # and return that frame
    
    
print("\n\n\nALL PRE HARDWARE CONFIG AND DEFINITIONS COMPLETE. GOOD TO RUN... good luck :P\n\n\n")

In [ ]:
# LOAD AND GET DIAGNOSTICS FROM THE OVERLAY


## CHANGE FILE HERE FOR BITSTREAM. or don't. ##
overlay_file = "pcam_rgb_224bb.bit"

ol = _LOAD_OVERLAY(overlay_file)

# Initial var setup for functions later.
#VDMA
vdma_base = ol.ip_dict['vdma_downscale']['phys_addr']
vdma = MMIO(vdma_base, 0x10000)
width_vdma, height_vdma = 224, 224 #could be 224

vdma_base_color = ol.ip_dict['vdma_color']['phys_addr']
vdma_color = MMIO(vdma_base_color, 0x10000)
width_vdma_color, height_vdma_color = 1920, 1080 #could be 224


# Set up IIC (actually this one just is the setup)
iic = AxiIIC(ol.ip_dict['axi_iic_0']) 
CAM_I2C_ADDR = 0x3C  # Standard 7-bit I2C address for the OV5640


#Camera power pin (adjust accordingly)
cam_pwup = ol.axi_gpio_0.channel1[0]


#Demosaic assign
demosaic = ol.vdemo
#INIT BELOW

# Initialize camera for 1080 at 30fps but do NOT start
_PCAM_PC(cam_pwup)           # give it the GPIO pin corresponding to the power/en
_PCAM_CONFIGURE_BASE()
_PCAM_CONFIGURE_1080p30()
## PCAM SHOULD BE CONFIGURED BUT *NOT* RUNNING



# Configure demosaic
_DEMOSAIC_CONFIG_RES(demosaic, 1920, 1080)
_DEMOSAIC_SET_BAYER(demosaic, 0x03)       # 0 = RGGB, 1 = GRBG, 2 = GBRG, 3 = BGGR (this one but colors are reversed so its actuall RGGB)
_DEMOSAIC_SET_CTRL(demosaic, 0x81)

#create frame buffer
frame_buffer1 = _ALLOCATE_FRAME_BUFFER(width_vdma, height_vdma)
frame_buffer2 = _ALLOCATE_FRAME_BUFFER(width_vdma, height_vdma)
frame_buffer3 = _ALLOCATE_FRAME_BUFFER(width_vdma, height_vdma)
frame_buffer4 = _ALLOCATE_FRAME_BUFFER(width_vdma, height_vdma)
frame_buffer_color1 = _ALLOCATE_FRAME_BUFFER(width_vdma_color, height_vdma_color)
frame_buffer_color2 = _ALLOCATE_FRAME_BUFFER(width_vdma_color, height_vdma_color)


#configure the VDMA
_VDMA_CONFIG_WHOLISTIC(vdma, frame_buffer1, frame_buffer2, frame_buffer3, frame_buffer4, width_vdma, height_vdma)
_VDMA_CONFIG_WHOLISTIC(vdma_color, frame_buffer_color1, frame_buffer_color2, frame_buffer_color1, frame_buffer_color2, width_vdma_color, height_vdma_color)

#quick check
# _RUN_PIPELINE_DIAGNOSTICS()



# Start the camera - set the pipeline is motion

#ok big time now
_PCAM_START_CAPTURE()


#quick check
_RUN_PIPELINE_DIAGNOSTICS()


In [ ]:
# THIS IS THE BIG CHEESE. THIS CELL SHOULD BE RUN LAST. IT CAPTURES DATA 
# FROM THE VDMA BUFFERS, SENDS THE DOWNSCALED IMAGE TO THE SHARED MEMORY
# AND THEN DISPLAYS A 1080p FRAME TO THE DP (if enabled)

#RUN THIS ONLY ONCE - THIS VERSION IS SINGLE THREADED SO IS ACTUALLY KINDA SHIT
# CONSIDER ADDING MULTITHREADING SO ONE THREAD DOES THE 224^2 FILE STUFF AND THE 
# OTHER MANAGES THE DISPLAYPORT OUTPUT (ALSO NEED TO ADD REDUNDANCY INCASE DP 
# DISCONNECTS)

# Cheers!

dp = DisplayPort()

dp.configure(VideoMode(1920, 1080, 24), PIXEL_RGB)
print("DP configured for 1920x1080 @ 24hz... since thats the default??")

import os

while True:
    frame = cap_latest_frame(vdma) 
    

    np.save('/dev/shm/temp_rgb.npy', frame)
    os.rename('/dev/shm/temp_rgb.npy', '/dev/shm/latest_rgb.npy')
    
    
    display_bounding_box_1080()
    
    
    time.sleep(0.005)
    
    
    
    